In [ ]:
REPO_URL = "https://github.com/tiom4eg/lensless_imaging.git"
BATCH_SIZE = 4

METHODS = [
    ("FISTA", "fista", None, None),
    ("ADMM-100", "admm100", None, None),
    ("ADMM-100 + SR", "admm100_sr", None, None),
    ("Le-ADMM-20", "leadmm20", "MHDCSM/lensless-leadmm20_20260609_193725", "model_best.pth"),
    ("modular pre", "modular_pre", "MHDCSM/lensless-modular_pre_20260610_100629", "model_best.pth"),
    ("modular post", "modular_post", "MHDCSM/lensless-modular_post_20260610_035523", "model_best.pth"),
    ("modular pre+post (12 эп)", "modular_prepost", "MHDCSM/lensless-modular_prepost_20260609_003122", "checkpoint-epoch12.pth"),
    ("modular pre+post (best)", "modular_prepost", "MHDCSM/lensless-modular_prepost_ft_20260609_190232", "model_best.pth"),
]

In [2]:
import os
from pathlib import Path

if Path("src").is_dir() and Path("inference.py").exists():
    print("Running from an existing checkout.")
else:
    repo_dir = Path("lensless_imaging")
    if not repo_dir.exists():
        get_ipython().system('git clone -q {REPO_URL} {repo_dir}')
    os.chdir(repo_dir)
    get_ipython().system('pip install -q -r requirements.txt')
print("cwd:", Path.cwd())

Running from an existing checkout.
cwd: /home/tiom4eg/prog/ml_hse/dl_hw5


## Метрики

Заменяем только подготовку пары `(pred, target)`:

* сейчас - `roi_crop(recon).clamp(0, 1)`;
* раньше - `normalize_minmax(roi_crop(recon)).clamp(0, 1)`.

GT не меняется.

In [ ]:
import sys
sys.path.append(".")
import torch
from torchmetrics.functional import peak_signal_noise_ratio, structural_similarity_index_measure
import lpips as lpips_lib

from src.utils.image_utils import roi_crop, normalize_minmax

device = "cuda" if torch.cuda.is_available() else "cpu"

lpips_model = lpips_lib.LPIPS(net="vgg", verbose=False).to(device).eval()
for p in lpips_model.parameters():
    p.requires_grad_(False)


def compute(pred, target):
    with torch.no_grad():
        return {
            "PSNR": peak_signal_noise_ratio(pred, target, data_range=1.0, dim=(1, 2, 3)).item(),
            "SSIM": structural_similarity_index_measure(pred, target, data_range=1.0).item(),
            "MSE": torch.mean((pred - target) ** 2).item(),
            "LPIPS": lpips_model(pred * 2 - 1, target * 2 - 1).mean().item(),
        }


def honest_pair(recon, lensed):
    return roi_crop(recon).clamp(0.0, 1.0), roi_crop(lensed).clamp(0.0, 1.0)


def normalized_pair(recon, lensed):
    return normalize_minmax(roi_crop(recon)).clamp(0.0, 1.0), roi_crop(lensed).clamp(0.0, 1.0)

/home/tiom4eg/prog/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/tiom4eg/prog/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/tiom4eg/prog/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
from hydra import compose, initialize
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate

GlobalHydra.instance().clear()
with initialize(version_base=None, config_path="src/configs"):
    base_cfg = compose(config_name="inference", overrides=["model=admm100"])

dataset = instantiate(base_cfg.datasets.test)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Test images: {len(dataset)} | device: {device}")

Test images: 1500 | device: cuda


In [5]:
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm


def build_model(model_cfg, hf_repo, filename):
    GlobalHydra.instance().clear()
    with initialize(version_base=None, config_path="src/configs"):
        cfg = compose(config_name="inference", overrides=[f"model={model_cfg}"])
    model = instantiate(cfg.model).to(device).eval()
    if hf_repo is not None:
        ckpt = hf_hub_download(repo_id=hf_repo, filename=filename)
        state = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(state["state_dict"] if "state_dict" in state else state)
    return model


def run_method(model):
    honest_tot = {k: 0.0 for k in ("PSNR", "SSIM", "MSE", "LPIPS")}
    norm_tot = {k: 0.0 for k in honest_tot}
    n = 0
    for batch in loader:
        lensless = batch["lensless"].to(device)
        psf = batch["psf"].to(device)
        lensed = batch["lensed"].to(device)
        with torch.no_grad():
            recon = model(lensless=lensless, psf=psf)["recon"]
        bs = recon.shape[0]
        for tot, prep in ((honest_tot, honest_pair), (norm_tot, normalized_pair)):
            pred, target = prep(recon, lensed)
            for k, v in compute(pred, target).items():
                tot[k] += v * bs
        n += bs
    return ({k: v / n for k, v in honest_tot.items()}, {k: v / n for k, v in norm_tot.items()})


results = {}
for label, model_cfg, hf_repo, filename in tqdm(METHODS, desc="methods"):
    model = build_model(model_cfg, hf_repo, filename)
    honest, normalized = run_method(model)
    results[label] = {"honest": honest, "normalized": normalized}
    print(f"{label:26s} honest PSNR={honest['PSNR']:6.3f}  |  normalized PSNR={normalized['PSNR']:6.3f}")
    del model
    if device == "cuda":
        torch.cuda.empty_cache()


methods:   0%|          | 0/8 [00:00<?, ?it/s]

/home/tiom4eg/prog/.venv/lib/python3.13/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `peak_signal_noise_ratio` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `peak_signal_noise_ratio` from `torchmetrics.image` instead.
  _future_warning(


/home/tiom4eg/prog/.venv/lib/python3.13/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(



methods:  12%|█▎        | 1/8 [05:58<41:49, 358.49s/it]

FISTA                      honest PSNR= 7.199  |  normalized PSNR=10.982


...


methods:  88%|████████▊ | 7/8 [1:06:42<05:50, 350.85s/it]

modular pre+post (12 эп)   honest PSNR= 8.322  |  normalized PSNR=16.024



methods: 100%|██████████| 8/8 [1:09:11<00:00, 286.63s/it]


methods: 100%|██████████| 8/8 [1:09:11<00:00, 518.98s/it]

modular pre+post (best)    honest PSNR=10.551  |  normalized PSNR=17.529


In [6]:
cols = ("PSNR", "SSIM", "MSE", "LPIPS")
head = f"{'метод':26s} | " + "  ".join(f"{c:>7s}" for c in cols) + "  ||  " + "  ".join(f"{c:>7s}" for c in cols)
print(f"Полный тест: {len(dataset)} изображений\n")
print(" " * 29 + "без нормализации" + " " * 24 + "с min-max нормализацией")
print(head)
print("-" * len(head))
for label, r in results.items():
    h = r["honest"]; nm = r["normalized"]
    left = "  ".join(f"{h[c]:7.3f}" for c in cols)
    right = "  ".join(f"{nm[c]:7.3f}" for c in cols)
    print(f"{label:26s} | {left}  ||  {right}")

Полный тест: 1500 изображений

                             без нормализации                        с min-max нормализацией
метод                      |    PSNR     SSIM      MSE    LPIPS  ||     PSNR     SSIM      MSE    LPIPS
-------------------------------------------------------------------------------------------------------
FISTA                      |   7.199    0.103    0.233    0.724  ||   10.982    0.265    0.103    0.771
ADMM-100                   |   7.233    0.101    0.232    0.724  ||   11.686    0.279    0.086    0.752
ADMM-100 + SR              |  11.816    0.318    0.084    0.756  ||   11.390    0.299    0.092    0.754
Le-ADMM-20                 |   7.271    0.105    0.231    0.738  ||   12.232    0.283    0.072    0.776
modular pre                |   7.486    0.096    0.224    0.745  ||   13.542    0.322    0.051    0.786
modular post               |  11.442    0.298    0.089    0.747  ||   14.734    0.388    0.039    0.740
modular pre+post (12 эп)   |   8.322    0.28